Copias impresas y electrónicas de *Modelado y simulación en Python* están disponibles en [No Starch Press](https://nostarch.com/modeling-and-simulation-python) y [Bookshop.org](https://bookshop.org/p/books/modeling-and-simulation-in-python-allen-b-downey/17836697?ean=9781718502161) y [Amazon](https://amzn.to/3y9UxNb).

# Epidemiología

*Modelado y Simulación en Python*

Copyright 2021 Allen Downey

Licencia: [Creative Commons Atribución-No Comercial-CompartirIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [1]:
# install Pint if necessary

try:
    import pint
except ImportError:
    !pip install pint

In [2]:
# download modsim.py if necessary

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)
    
download('https://raw.githubusercontent.com/AllenDowney/' +
         'ModSimPy/master/modsim.py')

In [3]:
# import functions from modsim

from modsim import *

En este capítulo desarrollaremos un modelo de epidemia a medida que se propaga en un
población susceptible y utilizarlo para evaluar la eficacia de
posibles intervenciones.

Mi presentación del modelo en los próximos capítulos se basa en un excelente artículo de David Smith y Lang Moore, "The SIR Model for Spread of Disease", *Journal of Online Mathematics and its Applications*, diciembre de 2001, disponible en <http://modsimpy.com/sir>.

.
## La plaga de los estudiantes de primer año

Cada año en Olin College, alrededor de 90 nuevos estudiantes llegan al campus desde
en todo el país y el mundo. La mayoría llegan sanos y felices, pero normalmente al menos uno trae consigo algún tipo de enfermedad infecciosa. Unas semanas más tarde, como era de esperar, una fracción de la clase entrante contrae lo que llamamos la "plaga de los estudiantes de primer año".

En este capítulo presentamos un modelo bien conocido de enfermedad infecciosa,
el modelo Kermack-McKendrick y utilizarlo para explicar la progresión de
la enfermedad en el transcurso del semestre, predecir el efecto de
posibles intervenciones (como la inmunización) y diseñar la campaña de intervención más efectiva.

Hasta ahora hemos hecho nuestro propio modelado; es decir, hemos elegido físico
sistemas, identificó factores que parecen importantes y tomó decisiones
sobre cómo representarlos. En este capítulo comenzamos con un existente
modelarlo y aplicarle ingeniería inversa. En el camino, consideramos el modelado.
decisiones que se tomaron e identificar sus capacidades y
limitaciones.

## El modelo Kermack-McKendrick

El modelo Kermack-McKendrick (KM) es un ejemplo de un *modelo SIR*,
Llamado así porque representa tres categorías de personas:

- *S*: Personas que son "susceptibles", es decir, capaces de
    contraer la enfermedad si entran en contacto con alguien que
    está infectado.

- *I*: Personas que son "infecciosas", es decir, capaces de pasar
    junto con la enfermedad si entran en contacto con alguien
    susceptible.

- *R*: Personas que están "recuperadas". En la versión básica del
    En el modelo, las personas que se han recuperado se consideran ya no
    infeccioso e inmune a la reinfección. Éste es un modelo razonable para algunas enfermedades, pero no para otras, por lo que debería estar en la lista de supuestos a reconsiderar más adelante.

Pensemos en cómo cambia con el tiempo el número de personas en cada categoría. Supongamos que sabemos que las personas con la enfermedad son contagiosas durante un período de 4 días, en promedio.
Si 100 personas son infecciosas en un momento determinado e ignoramos el momento particular en que cada una se infectó, esperamos que aproximadamente 1 de cada 4 se recupere en un día determinado.

Dicho de otra manera, si el tiempo entre recuperaciones es de 4 días, la tasa de recuperación es de aproximadamente 0,25 recuperaciones por día, lo que denotaremos con la letra griega gamma, $\gamma$, o el nombre de variable `gamma`.

Si el número total de personas en la población es $N$ y la fracción actualmente infecciosa es $i$, el número total de recuperaciones que esperamos por día es $\gamma i N$.

Ahora pensemos en el número de nuevas infecciones. Supongamos que sabemos que cada persona susceptible entra en contacto con 1 persona cada 3 días, en promedio, de una manera que provocaría que se infectara si la otra persona estuviera infectada. Denotaremos esta tasa de contacto con la letra griega beta, $\beta$, o el nombre de la variable `beta`.

Probablemente no sea razonable suponer que conocemos $\beta$ antes de
tiempo, pero luego veremos cómo estimarlo en base a datos de brotes anteriores.

Si $s$ es la fracción de la población que es susceptible, $s N$ es
el número de personas susceptibles, $\beta s N$ es el número de contactos por día y $\beta s i N$ es el número de aquellos contactos en los que la otra persona es infecciosa.

En resumen:

- El número de recuperaciones que esperamos por día es $\gamma i N$; Al dividir por $N$ se obtiene la fracción de la población que se recupera en un día, que es $\gamma i$.

- El número de nuevas infecciones que esperamos por día es $\beta s i N$; Al dividir por $N$ se obtiene la fracción de la población que se infecta en un día, que es $\beta s i$.

El modelo KM supone que la población está cerrada; es decir, nadie
llega o sale, por lo que el tamaño de la población, $N$, es constante.

## Las ecuaciones de KM

Si tratamos el tiempo como una cantidad continua, podemos escribir diferencial
ecuaciones que describen las tasas de cambio para $s$, $i$ y $r$ (donde $r$ es la fracción de la población que se ha recuperado):

$$\begin{aligned}
\frac{ds}{dt} &= -\beta s i \\
\frac{di}{dt} &= \beta s i - \gamma i\\
\frac{dr}{dt} &= \gamma i\end{aligned}$$ 

Para evitar saturar las ecuaciones, dejo implícito que $s$ es función del tiempo, $s(t)$, y lo mismo para $i$ y $r$.

Los modelos SIR son ejemplos de *modelos compartimentales*, llamados así porque
dividen el mundo en categorías discretas, o compartimentos, y
describir las transiciones de un compartimento a otro. Los compartimentos son
también llamados *stocks* y las transiciones entre ellos se llaman
*fluye*.

En este ejemplo, hay tres poblaciones: susceptible, infecciosa y
recuperados---y dos flujos---nuevas infecciones y recuperaciones. Compartimento
Los modelos a menudo se representan visualmente mediante diagramas de stock y flujo (ver <http://modsimpy.com/stock>).

La siguiente figura muestra el stock y diagrama de flujo del modelo KM.

![Stock y diagrama de flujo para un SIR
modelo.](https://github.com/AllenDowney/ModSim/raw/main/figs/stock_flow1.png)

Las acciones se representan con rectángulos y los flujos con flechas. El widget en el medio de las flechas representa una válvula que controla el caudal; el diagrama muestra los parámetros que controlan las válvulas.

## Implementando el modelo KM

Para un sistema físico dado, hay muchos modelos posibles, y para un
modelo dado, hay muchas maneras de representarlo. Por ejemplo, podemos
representar un modelo SIR como un diagrama de stock y flujo, como un conjunto de
ecuaciones diferenciales, o como un programa Python. El proceso de
representar un modelo en estas formas se llama *implementación*. en
En esta sección, implementamos el modelo KM en Python.

Representaré el estado inicial del sistema usando un objeto `State`.
con variables de estado `s`, `i` y `r`; representan la fracción de
la población de cada compartimento.

Podemos inicializar el objeto `State` con el *número* de personas en cada compartimento; por ejemplo, aquí está el estado inicial con un estudiante infectado en una clase de 90:

In [4]:
init = State(s=89, i=1, r=0)
show(init)

Podemos convertir los números a fracciones dividiendo por el total:

In [5]:
init /= init.sum()
show(init)

Por ahora, supongamos que conocemos el tiempo entre contactos y el tiempo entre
recuperaciones:

In [6]:
tc = 3             # time between contacts in days 
tr = 4             # recovery time in days

Podemos usarlos para calcular los parámetros del modelo:

In [7]:
beta = 1 / tc      # contact rate in per day
gamma = 1 / tr     # recovery rate in per day

Usaré un objeto `System` para almacenar los parámetros y las iniciales.
condiciones. La siguiente función toma los parámetros del sistema y devuelve un nuevo objeto `System`:

In [8]:
def make_system(beta, gamma):
    init = State(s=89, i=1, r=0)
    init /= init.sum()

    return System(init=init, t_end=7*14,
                  beta=beta, gamma=gamma)

El valor predeterminado para `t_end` es 14 semanas, aproximadamente la duración de un
semestre.

Así es como se ve el objeto `System`.

In [9]:
system = make_system(beta, gamma)
show(system)

Ahora que tenemos un objeto para representar el sistema y su estado, estamos listos para la función de actualización.

## La función de actualización

El propósito de una función de actualización es tomar el estado actual de un sistema y calcular el estado durante el siguiente paso de tiempo.
Aquí está la función de actualización que usaremos para el modelo KM:

In [10]:
def update_func(t, state, system):
    s, i, r = state.s, state.i, state.r

    infected = system.beta * i * s    
    recovered = system.gamma * i
    
    s -= infected
    i += infected - recovered
    r += recovered
    
    return State(s=s, i=i, r=r)

`update_func` toma como parámetros la hora actual, un objeto `State` y un objeto `System`.

La primera línea descomprime el objeto `State`, asignando los valores de las variables de estado a nuevas variables con los mismos nombres.
Este es un ejemplo de *asignación múltiple*.
El lado izquierdo es una secuencia de variables; el lado derecho es una secuencia de expresiones.
Los valores del lado derecho se asignan a las variables del lado izquierdo, en orden.
Al crear estas variables evitamos repetir `state` varias veces, lo que hace que el código sea más fácil de leer.

La función de actualización calcula `infected` y `recovered` como una fracción de la población, luego actualiza `s`, `i` y `r`. El valor de retorno es `State` que contiene los valores actualizados.

Podemos llamar a `update_func` así:

In [11]:
state = update_func(0, init, system)
show(state)

El resultado es el nuevo objeto `State`.

Es posible que observe que esta versión de `update_func` no utiliza uno de sus parámetros, `t`. Lo incluyo de todos modos porque las funciones de actualización
A veces dependen del tiempo, y conviene que todos tomen los mismos parámetros, los necesiten o no.

## Ejecutando la simulación

Ahora podemos simular el modelo en una secuencia de pasos de tiempo:

In [12]:
def run_simulation1(system, update_func):
    state = system.init

    for t in range(0, system.t_end):
        state = update_func(t, state, system)

    return state

Los parámetros de `run_simulation` son el objeto `System` y el
función de actualización. El objeto `System` contiene los parámetros, inicial
condiciones y valores de `0` y `t_end`.

Podemos llamar a `run_simulation` así:

In [13]:
final_state = run_simulation1(system, update_func)
show(final_state)

El resultado indica que después de 14 semanas (98 días), alrededor del 52% de los
La población sigue siendo susceptible, lo que significa que nunca fueron infectados.
casi el 48 % se ha recuperado, lo que significa que se infectó en algún momento, y menos del 1 % está infectado activamente.

## Recopilación de resultados

La versión anterior de `run_simulation` devuelve solo el estado final,
pero es posible que queramos ver cómo cambia el estado con el tiempo. Consideraremos dos formas de hacerlo: primero, usando tres objetos `TimeSeries` y luego usando un nuevo objeto llamado `TimeFrame`.

Aquí está la primera versión:

In [14]:
def run_simulation2(system, update_func):
    S = TimeSeries()
    I = TimeSeries()
    R = TimeSeries()

    state = system.init
    S[0], I[0], R[0] = state
    
    for t in range(0, system.t_end):
        state = update_func(t, state, system)
        S[t+1], I[t+1], R[t+1] = state.s, state.i, state.r
    
    return S, I, R

Primero, creamos objetos `TimeSeries` para almacenar los resultados.
A continuación inicializamos `state` y los primeros elementos de `S`, `I` y
`R`.

Dentro del bucle, usamos `update_func` para calcular el estado del sistema en el siguiente paso de tiempo, luego usamos asignación múltiple para descomprimir los elementos de `state`, asignando cada uno al `TimeSeries` correspondiente.

Al final de la función, devolvemos los valores `S`, `I` y `R`. Este es el primer ejemplo que hemos visto donde una función devuelve más de un valor.

Podemos ejecutar la función así:

In [15]:
S, I, R = run_simulation2(system, update_func)

Usaremos la siguiente función para trazar los resultados:

In [16]:
def plot_results(S, I, R):
    S.plot(style='--', label='Susceptible')
    I.plot(style='-', label='Infected')
    R.plot(style=':', label='Recovered')
    decorate(xlabel='Time (days)',
             ylabel='Fraction of population')

Y ejecútelo así: 

In [17]:
plot_results(S, I, R)

Se necesitan unas tres semanas (21 días) para que el brote comience y unas cinco semanas (35 días) para alcanzar su punto máximo. La fracción de la población que está infectada nunca es muy alta, pero suma. En total, casi la mitad de la población enferma.

## Ahora con un marco de tiempo

Si el número de variables de estado es pequeño, almacenarlas como separadas
Es posible que los objetos `TimeSeries` no sean tan malos. Pero una mejor alternativa es usar un `TimeFrame`, que es otro objeto definido en ModSim.
biblioteca.
Un `TimeFrame` es una especie de `DataFrame`, que utilizamos anteriormente para almacenar estimaciones de la población mundial.

Aquí hay una versión más concisa de `run_simulation` usando un `TimeFrame`:

In [18]:
def run_simulation(system, update_func):
    frame = TimeFrame(columns=system.init.index)
    frame.loc[0] = system.init
    
    for t in range(0, system.t_end):
        frame.loc[t+1] = update_func(t, frame.loc[t], system)
    
    return frame

La primera línea crea un `TimeFrame` vacío con una columna para cada
variable de estado. Luego, antes de que comience el ciclo, almacenamos el valor inicial.
condiciones en `TimeFrame` en `0`. Basado en la forma en que hemos estado usando
Objetos `TimeSeries`, es tentador escribir:

```
frame[0] = system.init
```

Pero cuando usa el operador de corchetes con `TimeFrame` o `DataFrame`, selecciona una columna, no una fila. 
Para seleccionar una fila, tenemos que usar `loc`, así:

```
frame.loc[0] = system.init
```

Dado que el valor del lado derecho es `State`, la asignación coincide
subir el índice del `State` con las columnas del `TimeFrame`; eso
es decir, asigna el valor `s` de `system.init` a la columna `s` de
`frame`, y también con `i` y `r`.

Cada vez que recorremos el ciclo, asignamos el `State` que obtenemos de `update_func` a la siguiente fila de `frame`.
Al final, devolvemos `frame`. 

Podemos llamar a esta versión de `run_simulation` así:

In [19]:
results = run_simulation(system, update_func)

Aquí están las primeras filas de los resultados.

In [20]:
results.head()

Las columnas de `TimeFrame` corresponden a las variables de estado, `s`, `i` y `r`.
Al igual que con `DataFrame`, podemos usar el operador de punto para seleccionar columnas.
desde un `TimeFrame`, por lo que podemos trazar los resultados de esta manera:

In [21]:
plot_results(results.s, results.i, results.r)

Los resultados son los mismos que antes, ahora en una forma más cómoda.

## Resumen

Este capítulo presenta un modelo SIR de enfermedades infecciosas y dos formas de recopilar los resultados, utilizando varios objetos `TimeSeries` o un único `TimeFrame`.
En el próximo capítulo usaremos el modelo para explorar el efecto de la inmunización.

Pero primero quizás quieras trabajar en estos ejercicios.

## Ejercicios

Este capítulo está disponible como un cuaderno Jupyter donde puede leer el texto, ejecutar el código y trabajar en los ejercicios. 
Puedes acceder a los cuadernos en <https://allendowney.github.io/ModSimPy/>.

### Ejercicio 1  

Supongamos que el tiempo entre contactos es de 4 días y el tiempo de recuperación es de 5 días.  Después de 14 semanas, ¿cuántos estudiantes en total han sido infectados?

Pista: ¿cuál es el cambio en `S` entre el principio y el final de la simulación?

In [22]:
# Solution goes here

In [23]:
# Solution goes here

In [24]:
# Solution goes here